In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import jenkspy
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
def load_and_prepare_data(train_path, synth_path, encoding='gbk'):
   
    train_df = pd.read_csv(train_path, encoding=encoding)
    synth_df = pd.read_csv(synth_path, encoding=encoding)
    
    exclude_cols = ['original_risk_label', 'Tag', 'abundance']
    feature_names = [col for col in train_df.columns if col not in exclude_cols]
    
    X_train = train_df[feature_names].values
    common_feats = [f for f in feature_names if f in synth_df.columns]
    X_synth = synth_df[common_feats].values
    X_combined = np.vstack([X_train, X_synth])
    n_original = len(train_df)
    n_synth = len(synth_df)
    abundance_train = train_df['abundance'].values
    if 'abundance' in synth_df.columns:
        abundance_synth = synth_df['abundance'].values
    else:
        abundance_synth = np.zeros(n_synth)
    
    
    return (feature_names, X_combined, X_train, X_synth,
            n_original, abundance_train, abundance_synth, train_df)

In [ ]:
def train_autoencoder(X_combined, encoding_dim=12, epochs=200, batch_size=64):

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_combined)
    
    input_dim = X_scaled.shape[1]
    input_layer = layers.Input(shape=(input_dim,))
    
    enc = layers.Dense(128, activation='relu')(input_layer)
    enc = layers.Dropout(0.3)(enc)
    enc = layers.Dense(64, activation='relu')(enc)
    enc = layers.Dropout(0.2)(enc)
    enc = layers.Dense(32, activation='relu')(enc)
    bottleneck = layers.Dense(encoding_dim, activation='relu', name='bottleneck')(enc)
    
    dec = layers.Dense(32, activation='relu')(bottleneck)
    dec = layers.Dropout(0.2)(dec)
    dec = layers.Dense(64, activation='relu')(dec)
    dec = layers.Dropout(0.3)(dec)
    dec = layers.Dense(128, activation='relu')(dec)
    output = layers.Dense(input_dim, activation='linear')(dec)
    
    autoencoder = Model(inputs=input_layer, outputs=output)
    autoencoder.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=200, restore_best_weights=True, min_delta=0.001
    )
    autoencoder.fit(
        X_scaled, X_scaled,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        verbose=1,
        callbacks=[early_stopping]
    )
    
    encoder_model = Model(inputs=input_layer, outputs=bottleneck)
    latent = encoder_model.predict(X_scaled, verbose=0)
    
    
    return scaler, encoder_model, latent, autoencoder

In [ ]:
def compute_risk_direction(latent_representation):

    pca = PCA(n_components=1)
    pca.fit(latent_representation)
    risk_direction = pca.components_[0]  
    risk_direction = risk_direction / np.linalg.norm(risk_direction)
    risk_property = np.dot(latent_representation, risk_direction)
    
    return risk_direction, risk_property

In [ ]:
def process_abundance_and_scale(abundance_train, abundance_synth, risk_property):

    abundance_combined = np.concatenate([abundance_train, abundance_synth])
    log_abundance = np.log10(abundance_combined + 1)
    risk_property_min = np.min(risk_property)
    risk_property_max = np.max(risk_property)
    log_abundance_min = np.min(log_abundance)
    log_abundance_max = np.max(log_abundance)
    risk_property_scaled = (risk_property - risk_property_min) / (risk_property_max - risk_property_min)
    log_abundance_scaled = (log_abundance - log_abundance_min) / (log_abundance_max - log_abundance_min)
    
    return (risk_property_min, risk_property_max,
            log_abundance, log_abundance_min, log_abundance_max,
            risk_property_scaled, log_abundance_scaled)

In [ ]:
def identify_healthy_cluster(risk_property_scaled, log_abundance_scaled, n_clusters=4, random_state=42):

    health_features = np.column_stack([risk_property_scaled, log_abundance_scaled])
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(health_features)
    cluster_centers = kmeans.cluster_centers_
    distances_to_origin = np.sqrt(np.sum(cluster_centers**2, axis=1))
    healthy_cluster_idx = np.argmin(distances_to_origin)
    healthy_indices = np.where(labels == healthy_cluster_idx)[0]
    
    return healthy_indices, kmeans, labels

In [ ]:
def fit_gmm_and_compute_distances(risk_property_scaled, log_abundance_scaled, healthy_indices, weights=(1, 2)):

    all_samples = np.column_stack([risk_property_scaled, log_abundance_scaled])
    healthy_samples = all_samples[healthy_indices]
    gmm = GaussianMixture(n_components=1, covariance_type='tied', random_state=42)
    gmm.fit(healthy_samples)
    weights = np.array(weights)
    mean = gmm.means_[0]
    weighted_distances = np.array([
        np.sqrt(np.sum((weights * (sample - mean))**2))
        for sample in all_samples
    ])
    log_probs = gmm.score_samples(all_samples)
    
    return gmm, weighted_distances, log_probs

In [ ]:
def assign_risk_levels(weighted_distances, log_probs, healthy_indices, n_original=None):

    healthy_log_probs = log_probs[healthy_indices]
    if len(healthy_log_probs) >= 2:
        breaks = jenkspy.jenks_breaks(healthy_log_probs, n_classes=2)
        log_prob_threshold = breaks[1]
    else:
        log_prob_threshold = np.min(healthy_log_probs) - 1e-6
    
    risk_levels = np.zeros(len(log_probs), dtype=int)
    risk_levels[:] = -1
    for i in range(len(log_probs)):
        if log_probs[i] >= log_prob_threshold:
            risk_levels[i] = 0
    
    non_healthy_indices = np.where(risk_levels == -1)[0]
    non_healthy_dist = weighted_distances[non_healthy_indices]
    non_healthy_breaks = None
    if len(non_healthy_dist) > 0:
        quantiles = [0, 0.4, 0.7, 0.9, 1.0]
        non_healthy_breaks = np.quantile(non_healthy_dist, quantiles)
        for idx in non_healthy_indices:
            d = weighted_distances[idx]
            if d < non_healthy_breaks[1]:
                risk_levels[idx] = 1
            elif d < non_healthy_breaks[2]:
                risk_levels[idx] = 2
            elif d < non_healthy_breaks[3]:
                risk_levels[idx] = 3
            else:
                risk_levels[idx] = 4
    else:
        print("Warning: No unhealthy samples, cannot classify levels 1 to 4")
    
    if n_original is not None:
        risk_levels = risk_levels[:n_original]
    
    return risk_levels, log_prob_threshold, non_healthy_breaks

In [ ]:
def build_result_dataframe(train_df, feature_names, risk_property, risk_property_scaled,
                           weighted_distances, risk_levels, abundance,
                           log_abundance, log_abundance_scaled, log_probs,
                           n_original=None):

    n_target = len(train_df)
    if n_original is not None:
        n_target = n_original
    
    def align_length(arr):
        if len(arr) > n_target:
            return arr[:n_target]
        return arr

    risk_property = align_length(risk_property)
    risk_property_scaled = align_length(risk_property_scaled)
    weighted_distances = align_length(weighted_distances)
    risk_levels = align_length(risk_levels)
    abundance = align_length(abundance)
    log_abundance = align_length(log_abundance)
    log_abundance_scaled = align_length(log_abundance_scaled)
    log_probs = align_length(log_probs)

    result_df = pd.DataFrame({
        'property_value': risk_property,
        'property_value_scaled': risk_property_scaled,
        'risk_distance': weighted_distances,
        'new_risk_level': risk_levels,
        'abundance': abundance,
        'log_abundance': log_abundance,
        'log_abundance_scaled': log_abundance_scaled,
        'health_probability': log_probs
    })
    
    for col in feature_names:
        if col in train_df.columns:
            result_df[col] = train_df[col].values[:n_target]
    
    return result_df

In [ ]:
feature_names, X_combined, X_train, X_synth, n_orig, ab_train, ab_synth, train_df = load_and_prepare_data('train.csv', 'xxx.csv')
scaler, encoder_model, latent, autoencoder = train_autoencoder(X_combined)
risk_direction, risk_property = compute_risk_direction(latent)
(risk_property_min, risk_property_max,
 log_abundance, log_abundance_min, log_abundance_max,
 risk_property_scaled, log_abundance_scaled) = process_abundance_and_scale(ab_train, ab_synth, risk_property)
healthy_indices, kmeans, labels = identify_healthy_cluster(risk_property_scaled, log_abundance_scaled, n_clusters=5)
gmm, weighted_distances, log_probs = fit_gmm_and_compute_distances(risk_property_scaled, log_abundance_scaled, healthy_indices)
risk_levels, threshold, non_healthy_breaks = assign_risk_levels(weighted_distances, log_probs, healthy_indices, n_orig)
result_df = build_result_dataframe(train_df, feature_names, risk_property, risk_property_scaled,
                                   weighted_distances, risk_levels, ab_train,
                                   log_abundance, log_abundance_scaled, log_probs, n_orig)
result_df.to_csv('xxx.csv', index=False, encoding='utf-8-sig')

In [ ]:
def predict_test_set(test_path, feature_names, scaler, encoder_model, risk_direction,
                     risk_property_min, risk_property_max,
                     log_abundance_min, log_abundance_max,
                     gmm, weights, log_prob_threshold, non_healthy_breaks,
                     encoding='gbk'):
   
    test_df = pd.read_csv(test_path, encoding=encoding)
    missing_feats = [f for f in feature_names if f not in test_df.columns]
    if missing_feats:
        raise ValueError(f"Missing feature columns in test set: {missing_feats}")
    X_test = test_df[feature_names].values
    X_test_scaled = scaler.transform(X_test)
    latent_test = encoder_model.predict(X_test_scaled, verbose=0)
    risk_property_test = np.dot(latent_test, risk_direction)
    if 'abundance' not in test_df.columns:
        raise ValueError("Missing 'abundance' column in test set")
    abundance_test = test_df['abundance'].values
    log_abundance_test = np.log10(abundance_test + 1)
    risk_property_scaled_test = (risk_property_test - risk_property_min) / (risk_property_max - risk_property_min)
    log_abundance_scaled_test = (log_abundance_test - log_abundance_min) / (log_abundance_max - log_abundance_min)
    test_samples = np.column_stack([risk_property_scaled_test, log_abundance_scaled_test])
    weights = np.array(weights)
    mean = gmm.means_[0]
    weighted_distances_test = np.array([
        np.sqrt(np.sum((weights * (sample - mean))**2))
        for sample in test_samples
    ])
    log_probs_test = gmm.score_samples(test_samples)
    risk_levels_test = np.zeros(len(test_samples), dtype=int)
    risk_levels_test[:] = -1  
    
    for i in range(len(test_samples)):
        if log_probs_test[i] >= log_prob_threshold:
            risk_levels_test[i] = 0  
        else:
            if non_healthy_breaks is not None and len(non_healthy_breaks) == 5:
                dist = weighted_distances_test[i]
                if dist < non_healthy_breaks[1]:
                    risk_levels_test[i] = 1
                elif dist < non_healthy_breaks[2]:
                    risk_levels_test[i] = 2
                elif dist < non_healthy_breaks[3]:
                    risk_levels_test[i] = 3
                else:
                    risk_levels_test[i] = 4
            else:
                risk_levels_test[i] = 1
    
    test_result_df = pd.DataFrame({
        'property_value': risk_property_test,
        'property_value_scaled': risk_property_scaled_test,
        'risk_distance': weighted_distances_test,
        'new_risk_level': risk_levels_test,
        'abundance': abundance_test,
        'log_abundance': log_abundance_test,
        'log_abundance_scaled': log_abundance_scaled_test,
        'health_probability': log_probs_test
    })
    
    for col in feature_names:
        if col in test_df.columns:
            test_result_df[col] = test_df[col].values
    
    # 10. 打印分布统计
    risk_counts = np.bincount(risk_levels_test.astype(int), minlength=5)
    risk_labels = ['low risk', 'moderate risk', 'high risk', 'very high risk', 'extremely high risk']
    for label, count in zip(risk_labels, risk_counts):
        print(f"{label}: {count} samples")
    
    return test_result_df

In [ ]:
test_result_df = predict_test_set(
    test_path='test.csv',
    feature_names=feature_names,
    scaler=scaler,
    encoder_model=encoder_model,
    risk_direction=risk_direction,
    risk_property_min=risk_property_min,
    risk_property_max=risk_property_max,
    log_abundance_min=log_abundance_min,
    log_abundance_max=log_abundance_max,
    gmm=gmm,
    weights=(1, 2),
    log_prob_threshold=threshold,
    non_healthy_breaks=non_healthy_breaks,
    encoding='gbk'
)

test_result_df.to_csv('test_risk_assessment.csv', index=False, encoding='utf-8-sig')